In [1]:
!pip install onnx onnxruntime skl2onnx

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.2 MB 3.0 MB/s eta 0:00:06
   --- ------------------------------------ 1.6/17.2 MB 3.5 MB/s eta 0:00:05
   ----- ---------------------------------- 2.4/17.2 MB 3.6 MB/s eta 0:00:05
   ------- -------------------------------- 3.1/17.2 MB 3.6 MB/s eta 0:00:04
   --------- ------------------------------ 4.2/17.2 MB 3.6 MB/s eta 0:00:04
   --------- ------------------------------ 4.2/17.2 MB 3.6 MB/s eta 0:00:04
   ------------ --------------------------- 5.2/17.2 MB 3.3 MB/s eta 0:00:04
   -------------- ------------------------- 6.0/17.2 MB 3.4 MB/s eta 0:00:04
   ---------------- ----------------------- 

In [9]:
!pip install onnxmltools

In [10]:
import sys
!{sys.executable} -m pip install onnxmltools

In [12]:
from pathlib import Path

OUTPUT_DIR = Path(r"C:\f1 new\data")
print(list(OUTPUT_DIR.glob("*")))

[WindowsPath('C:/f1 new/data/f1_gold.parquet'), WindowsPath('C:/f1 new/data/f1_laps_2022_2025.parquet'), WindowsPath('C:/f1 new/data/f1_laps_silver.parquet'), WindowsPath('C:/f1 new/data/f1_race_control.parquet'), WindowsPath('C:/f1 new/data/f1_results_2022_2025.parquet'), WindowsPath('C:/f1 new/data/f1_schedule_2022_2025.csv'), WindowsPath('C:/f1 new/data/reference.parquet'), WindowsPath('C:/f1 new/data/xgb_pit_model.pkl')]


In [15]:
import joblib 

OUTPUT_DIR = Path(r"C:\f1 new\data")
xgb = joblib.load(OUTPUT_DIR / "xgb_pit_model.pkl")


In [16]:
config = xgb.get_booster().save_config()
print("enable_categorical" in config)

import json
cfg = json.loads(config)
print(cfg["learner"]["gradient_booster"]["tree_train_param"])

False
{'alpha': '0', 'colsample_bylevel': '1', 'colsample_bynode': '1', 'colsample_bytree': '1', 'eta': '0.0500000007', 'gamma': '0', 'grow_policy': 'depthwise', 'interaction_constraints': '', 'lambda': '1', 'learning_rate': '0.0500000007', 'max_bin': '256', 'max_cat_threshold': '64', 'max_cat_to_onehot': '4', 'max_delta_step': '0', 'max_depth': '6', 'max_leaves': '0', 'min_child_weight': '1', 'min_split_loss': '0', 'monotone_constraints': '()', 'refresh_leaf': '1', 'reg_alpha': '0', 'reg_lambda': '1', 'sampling_method': 'uniform', 'sparse_threshold': '0.20000000000000001', 'subsample': '1'}


In [17]:
import xgboost
print(xgboost.__version__)

dump = xgb.get_booster().get_dump(dump_format="json")
print(dump[0][:800])

3.2.0
  { "nodeid": 0, "depth": 0, "split": "TyreAgePct", "split_condition": 0.31400001, "yes": 1, "no": 2, "missing": 2 , "children": [
    { "nodeid": 1, "depth": 1, "split": "DegradationSlope", "split_condition": 0.235300004, "yes": 3, "no": 4, "missing": 4 , "children": [
      { "nodeid": 3, "depth": 2, "split": "TyreAgePct", "split_condition": 0.229000002, "yes": 7, "no": 8, "missing": 8 , "children": [
        { "nodeid": 7, "depth": 3, "split": "CompoundEncoded", "split_condition": 4, "yes": 15, "no": 16, "missing": 16 , "children": [
          { "nodeid": 15, "depth": 4, "split": "LapNumber", "split_condition": 10, "yes": 31, "no": 32, "missing": 32 , "children": [
            { "nodeid": 31, "depth": 5, "split": "Position", "split_condition": 10, "yes": 63, "no": 64, "missing": 64 , "c


In [20]:
import json

dump = xgb.get_booster().get_dump(dump_format="json")

def check_node(node, tree_idx, path=""):
    if "children" in node and "split_condition" not in node:
        print(f"Tree {tree_idx} — Without split_condition:")
        print(json.dumps(node, indent=2)[:500])
        return True
    if "children" in node:
        for child in node["children"]:
            if check_node(child, tree_idx, path):
                return True
    return False

problem_found = False
for i, tree_str in enumerate(dump):
    tree = json.loads(tree_str)
    if check_node(tree, i):
        problem_found = True
        break

print("We found the problem", problem_found)

Tree 1 — Without split_condition:
{
  "nodeid": 22,
  "depth": 4,
  "split": "Rainfall",
  "yes": 45,
  "no": 46,
  "children": [
    {
      "nodeid": 45,
      "depth": 5,
      "split": "Humidity",
      "split_condition": 61,
      "yes": 85,
      "no": 86,
      "missing": 86,
      "children": [
        {
          "nodeid": 85,
          "leaf": 0.0881970897
        },
        {
          "nodeid": 86,
          "leaf": 0.143176526
        }
      ]
    },
    {
      "nodeid": 46,
      "leaf": -0.0496716462
    }
  ]
}
We found the problem True


In [33]:
import numpy as np 
import time 

X_single = np.random.rand(1, 14).astype(np.float32)

start = time.time()
for _ in range(10000):
    xgb.predict_proba(X_single)
xgb_single = (time.time() - start) / 10000 * 1000

print(f"XGBoost single row: {xgb_single:.3f} ms")
print(f"Monte Carlo 10,000 sim × 57 lap = {xgb_single * 10000 * 57 / 1000:.1f} sec")

XGBoost single row: 1.933 ms
Monte Carlo 10,000 sim × 57 lap = 1102.1 sec


In [36]:
import time 
import numpy as np 

X_all = np.random.rand(57, 14).astype(np.float32)

start = time.time() 

## for single * 57 
for _ in range(10000): 
    for i in range(57):
        xgb.predict_proba(X_all[i :i+1]) 
single_time = (time.time() - start)/10000 * 1000 

start = time.time() 

for _ in range(10000):
    xgb.predict_proba(X_all) 
batch_time = (time.time() - start)/10000 * 1000

print(f"Single × 57 : {single_time:.2f} ms")
print(f"Batch 57    : {batch_time:.2f} ms")
print(f"Speedup     : {single_time/batch_time:.1f}x")
print(f"\nMonte Carlo old  : {single_time * 10000 / 1000:.1f} sec")
print(f"Monte Carlo new  : {batch_time  * 10000 / 1000:.1f} sec") 

Single × 57 : 95.12 ms
Batch 57    : 2.44 ms
Speedup     : 38.9x

Monte Carlo old  : 951.2 sec
Monte Carlo new  : 24.4 sec


In [37]:
model = joblib.load(OUTPUT_DIR / "xgb_pit_model.pkl") 

In [40]:
import pandas as pd 
df = pd.read_parquet(OUTPUT_DIR / "f1_gold.parquet")

In [41]:
df.columns

Index(['Year', 'RoundNumber', 'Driver', 'Team', 'LapNumber', 'LapPct',
       'Compound', 'CompoundEncoded', 'TyreLife', 'TyreAgePct',
       'DegradationRate', 'DegradationSlope', 'Position', 'SCInNext3Laps',
       'TrackTemp', 'AirTemp', 'Humidity', 'WindSpeed', 'Rainfall',
       'WillPitNextLap'],
      dtype='object')

In [ ]:
def simulate_race(
        driver_laps : pd.DataFrame, 
        model, 
        feature_cols : list, 
        n_simulations :int = 10000
) -> dict : 
    pit_votes = np.zeros(len(driver_laps)) 
    x_all = driver_laps[feature_cols].values.astype(np.float32) 
    all_probs = model.predict_proba(x_all)[:, 1] 

    sc_mask = driver_laps['SCInNext3Laps'].values > 0 
    all_probs[sc_mask] = np.minimum(1.0, all_probs[sc_mask] * 1.5) 

    for _ in range(n_simulations):
        random_vals = np.random.rand(len(driver_laps))   
        pit_laps = np.where(random_vals < all_probs)[0] 
        if len(pit_laps) > 0 : 
            pit_votes[pit_laps[0]] += 1 

    pit_probability = pit_votes / n_simulations 
    best_lap_index = np.argmax(pit_probability) 
    best_lap = driver_laps.iloc[best_lap_index]['LapNumber']
    best_prob = pit_probability[best_lap_index] 

    return {
        'optimal_pit_lap' : int(best_lap),
        'probability'     : round(best_prob, 3),
        'pit_distribution': pit_probability,
        'lap_numbers'     : driver_laps['LapNumber'].values
    } 

In [60]:
import time
feature_cols = [
    'LapNumber', 'LapPct',
    'CompoundEncoded', 'TyreLife', 'TyreAgePct',
    'DegradationRate', 'DegradationSlope',
    'Position', 'SCInNext3Laps',
    'TrackTemp', 'AirTemp', 'Humidity', 'WindSpeed', 'Rainfall'
]

driver_laps = df[
    (df['Year'] == 2024) &
    (df['RoundNumber'] == 1) &
    (df['Driver'] == 'VER') & 
    (df['LapNumber'] >= 17)
].sort_values('LapNumber').reset_index(drop=True)

start   = time.time()
result  = simulate_race(driver_laps, xgb, feature_cols)
elapsed = time.time() - start

print(f"Optimal Pit Lap : {result['optimal_pit_lap']}")
print(f"Probability     : {result['probability']*100:.1f}%")
print(f"Time taken      : {elapsed:.2f} sec")

Optimal Pit Lap : 32
Probability     : 16.6%
Time taken      : 0.14 sec


In [61]:
print(df[
    (df['WillPitNextLap'] == 1) &
    (df['RoundNumber'] == 1) &
    (df['Year'] == 2024) &
    (df['Driver'] == 'VER')])

       Year  RoundNumber Driver             Team  LapNumber  LapPct Compound  \
45709  2024            1    VER  Red Bull Racing       16.0   0.281     SOFT   
45729  2024            1    VER  Red Bull Racing       36.0   0.632     HARD   

       CompoundEncoded  TyreLife  TyreAgePct  DegradationRate  \
45709                0      19.0       0.760            0.415   
45729                2      19.0       0.422            0.922   

       DegradationSlope  Position  SCInNext3Laps  TrackTemp  AirTemp  \
45709             0.073       1.0              0       23.2     18.1   
45729             0.000       1.0              0       22.6     18.0   

       Humidity  WindSpeed  Rainfall  WillPitNextLap  
45709      50.0        0.4     False               1  
45729      49.0        0.7     False               1  


In [62]:
model = joblib.load(OUTPUT_DIR / "xgb_pit_model.pkl") 
reference = pd.read_parquet(OUTPUT_DIR / "reference.parquet") 

COMPOUND_MAX_LIFE = {
    'SOFT': 25, 'MEDIUM': 35, 'HARD': 45,
    'INTERMEDIATE': 30, 'WET': 40
}
COMPOUND_ENCODING = {
    'SOFT': 0, 'MEDIUM': 1, 'HARD': 2,
    'INTERMEDIATE': 3, 'WET': 4
}

def build_inference_laps(
    round_number: int,
    compound    : str,
    tyre_life   : int,
    position    : int,
    lap_start   : int = 1,      
    track_temp  : float = None,
) -> pd.DataFrame:

    ref = reference[
        (reference['RoundNumber'] == round_number) &
        (reference['Compound']    == compound)
    ]
    if ref.empty:
        return pd.DataFrame()

    ref          = ref.iloc[0]
    total_laps   = int(ref['TotalLaps'])
    avg_slope    = ref['AvgDegradationSlope']
    sc_prob      = ref['SCProbability']
    avg_track    = track_temp if track_temp else ref['AvgTrackTemp']
    avg_air      = ref['AvgAirTemp']
    avg_humid    = ref['AvgHumidity']
    compound_max = COMPOUND_MAX_LIFE.get(compound, 30)

    laps = []
    current_tyre_life = tyre_life

    for lap_num in range(lap_start, total_laps + 1):
        tyre_age_pct = min(current_tyre_life / compound_max, 1.0)
        deg_rate     = avg_slope * current_tyre_life

        laps.append({
            'LapNumber'        : lap_num,
            'LapPct'           : round(lap_num / total_laps, 3),
            'CompoundEncoded'  : COMPOUND_ENCODING.get(compound, -1),
            'TyreLife'         : current_tyre_life,
            'TyreAgePct'       : round(tyre_age_pct, 3),
            'DegradationRate'  : round(deg_rate, 3),
            'DegradationSlope' : round(avg_slope, 4),
            'Position'         : position,
            'SCInNext3Laps'    : sc_prob,
            'TrackTemp'        : avg_track,
            'AirTemp'          : avg_air,
            'Humidity'         : avg_humid,
            'WindSpeed'        : 2.0,
            'Rainfall'         : False,
        })
        current_tyre_life += 1

    return pd.DataFrame(laps)

In [63]:
inference_laps = build_inference_laps(
    round_number = 1,
    compound     = 'HARD',
    tyre_life    = 1,
    position     = 1,
    lap_start    = 17,     
    track_temp   = 22.6
)

print(inference_laps[['LapNumber', 'TyreLife', 
                    'TyreAgePct', 'DegradationRate']].head(10))

   LapNumber  TyreLife  TyreAgePct  DegradationRate
0         17         1       0.022            0.023
1         18         2       0.044            0.046
2         19         3       0.067            0.070
3         20         4       0.089            0.093
4         21         5       0.111            0.116
5         22         6       0.133            0.139
6         23         7       0.156            0.162
7         24         8       0.178            0.185
8         25         9       0.200            0.209
9         26        10       0.222            0.232


In [64]:
start  = time.time()
result = simulate_race(inference_laps, xgb, feature_cols)
elapsed = time.time() - start

print(f"Optimal Pit Lap : {result['optimal_pit_lap']}")
print(f"Probability     : {result['probability']*100:.1f}%")
print(f"Time            : {elapsed:.2f} sec")

Optimal Pit Lap : 33
Probability     : 23.9%
Time            : 0.27 sec


In [66]:
def simulate_race(
        driver_laps : pd.DataFrame, 
        model, 
        feature_cols : list, 
        n_simulations :int = 10000
) -> dict : 
    x_all = driver_laps[feature_cols].values.astype(np.float32) 
    all_probs = model.predict_proba(x_all)[:, 1] 

    sc_mask = driver_laps['SCInNext3Laps'].values > 0 
    all_probs[sc_mask] = np.minimum(1.0, all_probs[sc_mask] * 1.5) 

    random_matrix = np.random.rand(n_simulations , len(driver_laps)) 
    success_matrix = random_matrix < all_probs 

    first_pit_laps = np.argmax(success_matrix , axis =1) 

    has_pit = np.any(success_matrix, axis = 1) 
    valid_pits = first_pit_laps[has_pit] 

    pit_votes = np.bincount(valid_pits, minlength=len(driver_laps)) 

    pit_probability = pit_votes / n_simulations 
    best_lap_index = np.argmax(pit_probability) 
    best_lap = driver_laps.iloc[best_lap_index]['LapNumber']
    best_prob = pit_probability[best_lap_index] 

    return {
        'optimal_pit_lap' : int(best_lap),
        'probability'     : round(best_prob, 3),
        'pit_distribution': pit_probability,
        'lap_numbers'     : driver_laps['LapNumber'].values
    } 

In [67]:
inference_laps = build_inference_laps(
    round_number = 1,
    compound     = 'HARD',
    tyre_life    = 1,
    position     = 1,
    lap_start    = 17,     
    track_temp   = 22.6
)

print(inference_laps[['LapNumber', 'TyreLife', 
                    'TyreAgePct', 'DegradationRate']].head(10))

   LapNumber  TyreLife  TyreAgePct  DegradationRate
0         17         1       0.022            0.023
1         18         2       0.044            0.046
2         19         3       0.067            0.070
3         20         4       0.089            0.093
4         21         5       0.111            0.116
5         22         6       0.133            0.139
6         23         7       0.156            0.162
7         24         8       0.178            0.185
8         25         9       0.200            0.209
9         26        10       0.222            0.232


In [68]:
start  = time.time()
result = simulate_race(inference_laps, xgb, feature_cols)
elapsed = time.time() - start

print(f"Optimal Pit Lap : {result['optimal_pit_lap']}")
print(f"Probability     : {result['probability']*100:.1f}%")
print(f"Time            : {elapsed:.2f} sec")

Optimal Pit Lap : 33
Probability     : 24.3%
Time            : 0.03 sec
